# E-commerce Sales Analysis — Step 3: SQL Business Questions

**Input:** `ecommerce_sales_clean.csv`
**Tool:** SQLite (via Python's built-in `sqlite3`) — no separate database server needed, but the SQL itself is standard and would run the same way on MySQL/PostgreSQL.

We load the cleaned data into a local SQLite database, then answer 20 realistic business questions purely in SQL. This is the kind of query set an interviewer will expect you to explain line by line.

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv('ecommerce_sales_clean.csv', parse_dates=['OrderDate'])

conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')

def q(sql):
    return pd.read_sql_query(sql, conn)

print("Table loaded. Row count check:")
print(q("SELECT COUNT(*) AS total_rows FROM orders"))

Table loaded. Row count check:
   total_rows
0       50088


### Q1. Total revenue and order count by year

In [2]:
result = q("""SELECT OrderYear,
       COUNT(*) AS total_orders,
       ROUND(SUM(NetAmount), 2) AS total_revenue
FROM orders
WHERE OrderStatus = 'Delivered'
GROUP BY OrderYear
ORDER BY OrderYear;""")
result

,OrderYear,total_orders,total_revenue
0,2023,14231,1.251025e+08
1,2024,14288,1.251278e+08


### Q2. Top 10 customers by total revenue

In [3]:
result = q("""SELECT CustomerID,
       COUNT(*) AS orders_placed,
       ROUND(SUM(NetAmount), 2) AS total_spent
FROM orders
WHERE OrderStatus = 'Delivered'
GROUP BY CustomerID
ORDER BY total_spent DESC
LIMIT 10;""")
result

,CustomerID,orders_placed,total_spent
0,CUST08760,6,140792.67
1,CUST03853,7,137572.08
2,CUST02245,11,131587.50
3,CUST01668,9,129705.33
4,CUST08428,12,127700.02
5,CUST00286,9,127346.35
6,CUST07534,8,125689.39
7,CUST08079,7,125010.42
8,CUST00415,8,124065.90
9,CUST07416,6,123132.61


### Q3. Revenue and order count by product category

In [4]:
result = q("""SELECT ProductCategory,
       COUNT(*) AS orders,
       ROUND(SUM(NetAmount), 2) AS revenue
FROM orders
WHERE OrderStatus = 'Delivered'
GROUP BY ProductCategory
ORDER BY revenue DESC;""")
result

,ProductCategory,orders,revenue
0,Fashion,4825,42751801.50
1,Books,4860,42292924.16
2,Home & Kitchen,4742,41586416.21
3,Beauty,4704,41321117.38
4,Sports,4655,41150873.91
5,Electronics,4733,41127182.49


### Q4. Top 10 best-selling products by quantity

In [5]:
result = q("""SELECT ProductName,
       SUM(Quantity) AS total_units_sold
FROM orders
WHERE OrderStatus = 'Delivered'
GROUP BY ProductName
ORDER BY total_units_sold DESC
LIMIT 10;""")
result

,ProductName,total_units_sold
0,Sports Cap,1267
1,Women's Kurti,1257
2,Curtains,1245
3,Blender,1224
4,Keyboard,1217
5,Poetry Collection,1206
6,Business Book,1201
7,Water Bottle,1194
8,Formal Shirt,1182
9,Cookbook,1182


### Q5. Monthly revenue trend

In [6]:
result = q("""SELECT OrderMonthName,
       ROUND(SUM(NetAmount), 2) AS revenue
FROM orders
WHERE OrderStatus = 'Delivered'
GROUP BY OrderMonthName, OrderYear, OrderMonth
ORDER BY OrderYear, OrderMonth;""")
result

,OrderMonthName,revenue
0,Jan-2023,11144577.50
1,Feb-2023,9598935.13
2,Mar-2023,10523992.05
3,Apr-2023,10152968.90
4,May-2023,11044003.43
5,Jun-2023,9396219.46
6,Jul-2023,11205263.80
7,Aug-2023,10759004.77
8,Sep-2023,9984826.24
9,Oct-2023,10480418.23


### Q6. Cancellation rate by product category

In [7]:
result = q("""SELECT ProductCategory,
       COUNT(*) AS total_orders,
       SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,
       ROUND(100.0 * SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancel_rate_pct
FROM orders
GROUP BY ProductCategory
ORDER BY cancel_rate_pct DESC;""")
result

,ProductCategory,total_orders,cancelled_orders,cancel_rate_pct
0,Electronics,8373,1227,14.65
1,Home & Kitchen,8379,1213,14.48
2,Beauty,8229,1181,14.35
3,Sports,8250,1181,14.32
4,Fashion,8338,1178,14.13
5,Books,8519,1174,13.78


### Q7. Return rate by product category

In [8]:
result = q("""SELECT ProductCategory,
       COUNT(*) AS total_orders,
       SUM(CASE WHEN OrderStatus = 'Returned' THEN 1 ELSE 0 END) AS returned_orders,
       ROUND(100.0 * SUM(CASE WHEN OrderStatus = 'Returned' THEN 1 ELSE 0 END) / COUNT(*), 2) AS return_rate_pct
FROM orders
GROUP BY ProductCategory
ORDER BY return_rate_pct DESC;""")
result

,ProductCategory,total_orders,returned_orders,return_rate_pct
0,Home & Kitchen,8379,1267,15.12
1,Sports,8250,1195,14.48
2,Electronics,8373,1199,14.32
3,Books,8519,1218,14.30
4,Beauty,8229,1156,14.05
5,Fashion,8338,1142,13.70


### Q8. New vs Returning customer revenue split

In [9]:
result = q("""SELECT CustomerType,
       COUNT(*) AS orders,
       ROUND(SUM(NetAmount), 2) AS revenue,
       ROUND(AVG(NetAmount), 2) AS avg_order_value
FROM orders
WHERE OrderStatus = 'Delivered'
GROUP BY CustomerType
ORDER BY revenue DESC;""")
result

,CustomerType,orders,revenue,avg_order_value
0,Returning,16577,1.453699e+08,8769.37
1,New,11122,9.752981e+07,8769.09
2,Unknown,820,7.330589e+06,8939.74


### Q9. Repeat customers — customers with more than 1 delivered order

In [10]:
result = q("""SELECT COUNT(*) AS repeat_customers
FROM (
    SELECT CustomerID
    FROM orders
    WHERE OrderStatus = 'Delivered'
    GROUP BY CustomerID
    HAVING COUNT(*) > 1
);""")
result

,repeat_customers
0,7458


### Q10. One-time vs repeat customer count (customer-level retention check)

In [11]:
result = q("""SELECT
    CASE WHEN order_count > 1 THEN 'Repeat Customer' ELSE 'One-Time Customer' END AS customer_segment,
    COUNT(*) AS num_customers
FROM (
    SELECT CustomerID, COUNT(*) AS order_count
    FROM orders
    WHERE OrderStatus = 'Delivered'
    GROUP BY CustomerID
)
GROUP BY customer_segment;""")
result

,customer_segment,num_customers
0,One-Time Customer,1185
1,Repeat Customer,7458


### Q11. Average order value (AOV) by payment method

In [12]:
result = q("""SELECT PaymentMethod,
       COUNT(*) AS orders,
       ROUND(AVG(NetAmount), 2) AS avg_order_value
FROM orders
WHERE OrderStatus = 'Delivered'
GROUP BY PaymentMethod
ORDER BY avg_order_value DESC;""")
result

,PaymentMethod,orders,avg_order_value
0,UPI,3385,8980.84
1,Credit Card,6788,8830.74
2,Debit Card,3440,8803.42
3,Unknown,4836,8734.11
4,Cash On Delivery,6676,8685.86
5,Net Banking,3394,8655.98


### Q12. Top 10 cities by revenue

In [13]:
result = q("""SELECT City,
       COUNT(*) AS orders,
       ROUND(SUM(NetAmount), 2) AS revenue
FROM orders
WHERE OrderStatus = 'Delivered' AND City != 'Unknown'
GROUP BY City
ORDER BY revenue DESC
LIMIT 10;""")
result

,City,orders,revenue
0,Surat,2629,23086884.50
1,Bengaluru,2590,22540593.40
2,Delhi,1313,11788588.73
3,Nagpur,1332,11705109.93
4,Hyderabad,1294,11700615.53
5,Lucknow,1329,11644990.89
6,Pune,1326,11637793.32
7,Patna,1300,11628359.21
8,Chennai,1287,11569418.57
9,Jaipur,1313,11446280.98


### Q13. Discount percent vs average quantity purchased (does discount drive volume?)

In [14]:
result = q("""SELECT DiscountPercent,
       COUNT(*) AS orders,
       ROUND(AVG(Quantity), 2) AS avg_quantity
FROM orders
WHERE OrderStatus = 'Delivered'
GROUP BY DiscountPercent
ORDER BY DiscountPercent;""")
result

,DiscountPercent,orders,avg_quantity
0,0.0,11006,2.36
1,5.0,3486,2.38
2,10.0,3512,2.36
3,15.0,3533,2.34
4,20.0,3482,2.38
5,25.0,3500,2.36


### Q14. Highest revenue-generating single product per category

In [15]:
result = q("""SELECT ProductCategory, ProductName, revenue FROM (
    SELECT ProductCategory, ProductName,
           ROUND(SUM(NetAmount), 2) AS revenue,
           RANK() OVER (PARTITION BY ProductCategory ORDER BY SUM(NetAmount) DESC) AS rnk
    FROM orders
    WHERE OrderStatus = 'Delivered'
    GROUP BY ProductCategory, ProductName
)
WHERE rnk = 1;""")
result

,ProductCategory,ProductName,revenue
0,Beauty,Moisturizer,4427782.90
1,Books,Poetry Collection,4499943.07
2,Electronics,Keyboard,4303053.54
3,Fashion,Women's Kurti,4959554.51
4,Home & Kitchen,Curtains,4793173.84
5,Sports,Yoga Mat,4279511.35


### Q15. Month-over-month revenue growth rate

In [16]:
result = q("""WITH monthly AS (
    SELECT OrderYear, OrderMonth, OrderMonthName,
           SUM(NetAmount) AS revenue
    FROM orders
    WHERE OrderStatus = 'Delivered'
    GROUP BY OrderYear, OrderMonth, OrderMonthName
)
SELECT OrderMonthName,
       ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY OrderYear, OrderMonth)) /
             LAG(revenue) OVER (ORDER BY OrderYear, OrderMonth), 2) AS mom_growth_pct
FROM monthly
ORDER BY OrderYear, OrderMonth;""")
result

,OrderMonthName,revenue,mom_growth_pct
0,Jan-2023,11144577.50,NaN
1,Feb-2023,9598935.13,-13.87
2,Mar-2023,10523992.05,9.64
3,Apr-2023,10152968.90,-3.53
4,May-2023,11044003.43,8.78
5,Jun-2023,9396219.46,-14.92
6,Jul-2023,11205263.80,19.25
7,Aug-2023,10759004.77,-3.98
8,Sep-2023,9984826.24,-7.20
9,Oct-2023,10480418.23,4.96


### Q16. Customers who never completed a delivered order (all cancelled/returned/pending)

In [17]:
result = q("""SELECT COUNT(DISTINCT CustomerID) AS customers_with_no_delivered_order
FROM orders
WHERE CustomerID NOT IN (
    SELECT DISTINCT CustomerID FROM orders WHERE OrderStatus = 'Delivered'
);""")
result

,customers_with_no_delivered_order
0,329


### Q17. Revenue contribution of top 10% customers (Pareto / 80-20 check)

In [18]:
result = q("""WITH customer_rev AS (
    SELECT CustomerID, SUM(NetAmount) AS total_spent
    FROM orders
    WHERE OrderStatus = 'Delivered'
    GROUP BY CustomerID
),
ranked AS (
    SELECT *, NTILE(10) OVER (ORDER BY total_spent DESC) AS decile
    FROM customer_rev
)
SELECT decile, COUNT(*) AS customers, ROUND(SUM(total_spent), 2) AS revenue
FROM ranked
GROUP BY decile
ORDER BY decile;""")
result

,decile,customers,revenue
0,1,865,62555809.38
1,2,865,43566461.24
2,3,865,35078366.11
3,4,864,28719182.10
4,5,864,23726064.54
5,6,864,19423594.90
6,7,864,15334270.99
7,8,864,11459221.30
8,9,864,7395153.07
9,10,864,2972192.02


### Q18. Average discount given per category

In [19]:
result = q("""SELECT ProductCategory,
       ROUND(AVG(DiscountPercent), 2) AS avg_discount_pct
FROM orders
GROUP BY ProductCategory
ORDER BY avg_discount_pct DESC;""")
result

,ProductCategory,avg_discount_pct
0,Sports,9.26
1,Books,9.26
2,Electronics,9.25
3,Home & Kitchen,9.24
4,Beauty,9.20
5,Fashion,9.14


### Q19. Orders by day of week (which days sell the most?)

In [20]:
result = q("""SELECT CASE CAST(strftime('%w', OrderDate) AS INTEGER)
            WHEN 0 THEN 'Sunday' WHEN 1 THEN 'Monday' WHEN 2 THEN 'Tuesday'
            WHEN 3 THEN 'Wednesday' WHEN 4 THEN 'Thursday' WHEN 5 THEN 'Friday'
            ELSE 'Saturday' END AS day_of_week,
       COUNT(*) AS orders
FROM orders
GROUP BY strftime('%w', OrderDate)
ORDER BY orders DESC;""")
result

,day_of_week,orders
0,Tuesday,7243
1,Sunday,7219
2,Monday,7158
3,Saturday,7133
4,Friday,7133
5,Wednesday,7120
6,Thursday,7082


### Q20. Overall churn proxy — % of customers with only cancelled/returned orders and no delivered order

In [21]:
result = q("""SELECT
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN CustomerID NOT IN
        (SELECT DISTINCT CustomerID FROM orders WHERE OrderStatus = 'Delivered')
        THEN CustomerID END) / COUNT(DISTINCT CustomerID), 2) AS churn_proxy_pct
FROM orders;""")
result

,churn_proxy_pct
0,3.67


## Wrap-up

These 20 queries cover revenue trends, customer segmentation (new/returning, repeat vs one-time, Pareto analysis),
cancellation/return diagnostics, discount effectiveness, and time-based patterns — the same categories of question
a Data Analyst interview or a real stakeholder request usually falls into.

Next step: bring the key numbers from here into an Excel KPI workbook and a Power BI dashboard.